In [15]:
import pandas as pd
import os
import matplotlib.pyplot as plt

In [16]:
df = pd.read_csv(os.path.join("../Merged_Olist_Data", "delivery_analysis.csv"))
print(df.shape)
df.head()

(98207, 22)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,review_id,review_score,...,review_creation_date,review_answer_timestamp,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,Days_Difference,Delivery_Status,product_id,product_category_name
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,a54f0611adc9ed256b57ede6b6eb5114,4.0,...,2017-10-11 00:00:00,2017-10-12 03:43:48,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,7.0,On Time,87285b34884572647811a353c7ac498a,utilidades_domesticas
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,8d5266042046a06655c8db133d120ba5,4.0,...,2018-08-08 00:00:00,2018-08-08 18:37:50,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,5.0,On Time,595fac2a385ac33a80bd5114aec74eb8,perfumaria
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,e73b67b67587f7644d5bd1a52deb1b01,5.0,...,2018-08-18 00:00:00,2018-08-22 19:07:58,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,17.0,On Time,aa4383b373c6aca5d8797843e5594415,automotivo
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,359d03e676b3c069f62cadba8dd3f6e8,5.0,...,2017-12-03 00:00:00,2017-12-05 19:21:58,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,12.0,On Time,d0b61bfb1de832b15ba9d266ca96e5b0,pet_shop
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,e50934924e227544ba8246aeb3770dd4,5.0,...,2018-02-17 00:00:00,2018-02-18 13:02:51,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,9.0,On Time,65266b2da20d04dbe00c5c2d3bb7859e,papelaria


### Calculate Late Delivery Rate Per State
We group orders by `customer_state` and calculate three things for each state:
- Total number of orders
- Number of late orders (Late + Super Late combined)
- Percentage of late orders out of total

This gives us a single comparable number per state so we can rank them.

In [17]:
df["is_late"] = df["Delivery_Status"].isin(["Late", "Super Late"])

# Group by state
state_summary = (
    df.groupby("customer_state")
    .agg(
        total_orders=("order_id", "count"),
        late_orders=("is_late", "sum")
    )
    .reset_index()
)

# Calculate % late
state_summary["percentage_of_late_deliveries"] = (
    state_summary["late_orders"] / state_summary["total_orders"] * 100
).round(2)

# Sort worst to best
state_summary = state_summary.sort_values("percentage_of_late_deliveries", ascending=False)

print(state_summary)

   customer_state  total_orders  late_orders  percentage_of_late_deliveries
1              AL           411           95                          23.11
9              MA           736          141                          19.16
16             PI           490           76                          15.51
5              CE          1323          196                          14.81
24             SE           345           51                          14.78
4              BA          3344          457                          13.67
18             RJ         12698         1664                          13.10
26             TO           278           35                          12.59
7              ES          2018          244                          12.09
13             PA           969          117                          12.07
11             MS           708           81                          11.44
21             RR            45            5                          11.11
14          

In [18]:
state_summary.to_csv(os.path.join("../Merged_Olist_Data", "state_late_rates.csv"), index=False)
print("state_late_rates.csv saved to outputs folder")

state_late_rates.csv saved to outputs folder


### Top 5 states with Late Deliveries

In [19]:
top5 = state_summary.head(5)

print("TOP 5 WORST PERFORMING STATES")
print("=" * 40)
for _, row in top5.iterrows():
    print(f"{row['customer_state']}: {row['percentage_of_late_deliveries']}% late ({int(row['late_orders'])} of {int(row['total_orders'])} orders)")

TOP 5 WORST PERFORMING STATES
AL: 23.11% late (95 of 411 orders)
MA: 19.16% late (141 of 736 orders)
PI: 15.51% late (76 of 490 orders)
CE: 14.81% late (196 of 1323 orders)
SE: 14.78% late (51 of 345 orders)


In [1]:
# Sort for display
"""
plot_data = state_summary.sort_values("percentage_of_late_deliveries", ascending=True)

# Assign colors by severity
colors = ["red" if x > 15 else "orange" if x > 8 else "green"
          for x in plot_data["percentage_of_late_deliveries"]]

# Plot
fig, ax = plt.subplots(figsize=(10, 10))

ax.barh(plot_data["customer_state"], plot_data["percentage_of_late_deliveries"], color=colors)
ax.set_xlabel("% Late Orders")
ax.set_ylabel("State")
ax.set_title("Veridi Logistics — Late Delivery Rate by Brazilian State (%)")

# Add value labels
for i, val in enumerate(plot_data["percentage_of_late_deliveries"]):
    ax.text(val + 0.2, i, f"{val:.1f}%", va="center", fontsize=9)

# Add legend
from matplotlib.patches import Patch
legend = [
    Patch(color="red", label="Critical (>15%)"),
    Patch(color="orange", label="Warning (8-15%)"),
    Patch(color="green", label="Good (<8%)")
]
ax.legend(handles=legend, loc="lower right")

plt.tight_layout()
plt.savefig(os.path.join("../Charts", "Late delivery rate by state.png"), dpi=150)
plt.show()
"""

'\nplot_data = state_summary.sort_values("percentage_of_late_deliveries", ascending=True)\n\n# Assign colors by severity\ncolors = ["red" if x > 15 else "orange" if x > 8 else "green"\n          for x in plot_data["percentage_of_late_deliveries"]]\n\n# Plot\nfig, ax = plt.subplots(figsize=(10, 10))\n\nax.barh(plot_data["customer_state"], plot_data["percentage_of_late_deliveries"], color=colors)\nax.set_xlabel("% Late Orders")\nax.set_ylabel("State")\nax.set_title("Veridi Logistics — Late Delivery Rate by Brazilian State (%)")\n\n# Add value labels\nfor i, val in enumerate(plot_data["percentage_of_late_deliveries"]):\n    ax.text(val + 0.2, i, f"{val:.1f}%", va="center", fontsize=9)\n\n# Add legend\nfrom matplotlib.patches import Patch\nlegend = [\n    Patch(color="red", label="Critical (>15%)"),\n    Patch(color="orange", label="Warning (8-15%)"),\n    Patch(color="green", label="Good (<8%)")\n]\nax.legend(handles=legend, loc="lower right")\n\nplt.tight_layout()\nplt.savefig(os.path.jo

In [2]:
"""
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import requests
import numpy as np

# Download Brazil GeoJSON
url = "https://raw.githubusercontent.com/codeforamerica/click_that_hood/master/public/data/brazil-states.geojson"
response = requests.get(url)
geojson = response.json()

# Color mapping
state_colors = {}
for _, row in state_summary.iterrows():
    if row["percentage_of_late_deliveries"] > 15:
        state_colors[row["customer_state"]] = "red"
    elif row["percentage_of_late_deliveries"] > 8:
        state_colors[row["customer_state"]] = "orange"
    else:
        state_colors[row["customer_state"]] = "green"

# Plot the map
fig, ax = plt.subplots(figsize=(14, 12))

for feature in geojson["features"]:
    sigla = feature["properties"]["sigla"]
    color = state_colors.get(sigla, "lightgrey")
    coords = feature["geometry"]["coordinates"]
    geo_type = feature["geometry"]["type"]

    # Collect all points to find centroid for label
    all_x, all_y = [], []

    if geo_type == "Polygon":
        coords = [coords]
    for polygon in coords:
        for ring in polygon:
            xs, ys = zip(*ring)
            ax.fill(xs, ys, color=color, edgecolor="white", linewidth=0.5)
            ax.plot(xs, ys, color="white", linewidth=0.5)
            all_x.extend(xs)
            all_y.extend(ys)

    # Calculate centroid and place state label
    if all_x and all_y:
        centroid_x = np.mean(all_x)
        centroid_y = np.mean(all_y)
        ax.text(
            centroid_x, centroid_y, sigla,
            ha="center", va="center",
            fontsize=7, fontweight="bold",
            color="white",
            bbox=dict(boxstyle="round,pad=0.1", facecolor="black", alpha=0.4)
        )

# Add legend
legend = [
    mpatches.Patch(color="red", label="Critical (>15%)"),
    mpatches.Patch(color="orange", label="Warning (8-15%)"),
    mpatches.Patch(color="green", label="Good (<8%)")
]
ax.legend(handles=legend, loc="lower left", fontsize=11)
ax.set_title("Veridi Logistics — Late Delivery Rate Across Brazil", fontsize=14)
ax.axis("off")

plt.tight_layout()
plt.savefig(os.path.join("../Charts", "brazil_map.png"), dpi=150)
plt.show()
print("Map saved to outputs folder")
"""

'\nimport matplotlib.pyplot as plt\nimport matplotlib.patches as mpatches\nimport requests\nimport numpy as np\n\n# Download Brazil GeoJSON\nurl = "https://raw.githubusercontent.com/codeforamerica/click_that_hood/master/public/data/brazil-states.geojson"\nresponse = requests.get(url)\ngeojson = response.json()\n\n# Color mapping\nstate_colors = {}\nfor _, row in state_summary.iterrows():\n    if row["percentage_of_late_deliveries"] > 15:\n        state_colors[row["customer_state"]] = "red"\n    elif row["percentage_of_late_deliveries"] > 8:\n        state_colors[row["customer_state"]] = "orange"\n    else:\n        state_colors[row["customer_state"]] = "green"\n\n# Plot the map\nfig, ax = plt.subplots(figsize=(14, 12))\n\nfor feature in geojson["features"]:\n    sigla = feature["properties"]["sigla"]\n    color = state_colors.get(sigla, "lightgrey")\n    coords = feature["geometry"]["coordinates"]\n    geo_type = feature["geometry"]["type"]\n\n    # Collect all points to find centroid 